<a href="https://www.kaggle.com/code/youssefhazem75/f1-prediction?scriptVersionId=355255505" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
circuits_df = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/circuits.csv', index_col=0, na_values='\\N')
circuits_df.info()
circuits_df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 77 entries, 1 to 79
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   circuitRef  77 non-null     object 
 1   name        77 non-null     object 
 2   location    77 non-null     object 
 3   country     77 non-null     object 
 4   lat         77 non-null     float64
 5   lng         77 non-null     float64
 6   alt         77 non-null     int64  
 7   url         77 non-null     object 
dtypes: float64(2), int64(1), object(5)
memory usage: 5.4+ KB


,circuitRef,name,location,country,lat,lng,alt,url
circuitId,,,,,,,,
1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.84970,144.96800,10,http://en.wikipedia.org/wiki/Melbourne_Grand_P...
2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.73800,18,http://en.wikipedia.org/wiki/Sepang_Internatio...
3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.03250,50.51060,7,http://en.wikipedia.org/wiki/Bahrain_Internati...
4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.57000,2.26111,109,http://en.wikipedia.org/wiki/Circuit_de_Barcel...
5,istanbul,Istanbul Park,Istanbul,Turkey,40.95170,29.40500,130,http://en.wikipedia.org/wiki/Istanbul_Park


In [3]:
constructor_results_df = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/constructor_results.csv', index_col=0, na_values='\\N')
constructor_results_df.info()
constructor_results_df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 12625 entries, 1 to 17129
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   raceId         12625 non-null  int64  
 1   constructorId  12625 non-null  int64  
 2   points         12625 non-null  float64
 3   status         17 non-null     object 
dtypes: float64(1), int64(2), object(1)
memory usage: 493.2+ KB


,raceId,constructorId,points,status
constructorResultsId,,,,
1,18,1,14.0,NaN
2,18,2,8.0,NaN
3,18,3,9.0,NaN
4,18,4,5.0,NaN
5,18,5,2.0,NaN


In [4]:
constructor_standings_df = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/constructor_standings.csv', index_col=0, na_values='\\N')
constructor_standings_df.info()
constructor_standings_df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 13391 entries, 1 to 28982
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   raceId         13391 non-null  int64  
 1   constructorId  13391 non-null  int64  
 2   points         13391 non-null  float64
 3   position       13391 non-null  int64  
 4   positionText   13391 non-null  object 
 5   wins           13391 non-null  int64  
dtypes: float64(1), int64(4), object(1)
memory usage: 732.3+ KB


,raceId,constructorId,points,position,positionText,wins
constructorStandingsId,,,,,,
1,18,1,14.0,1,1,1
2,18,2,8.0,3,3,0
3,18,3,9.0,2,2,0
4,18,4,5.0,4,4,0
5,18,5,2.0,5,5,0


In [5]:
constructors_df = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/constructors.csv', index_col=0, na_values='\\N')
constructors_df.info()
constructors_df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 212 entries, 1 to 215
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   constructorRef  212 non-null    object
 1   name            212 non-null    object
 2   nationality     212 non-null    object
 3   url             212 non-null    object
dtypes: object(4)
memory usage: 8.3+ KB


,constructorRef,name,nationality,url
constructorId,,,,
1,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren
2,bmw_sauber,BMW Sauber,German,http://en.wikipedia.org/wiki/BMW_Sauber
3,williams,Williams,British,http://en.wikipedia.org/wiki/Williams_Grand_Pr...
4,renault,Renault,French,http://en.wikipedia.org/wiki/Renault_in_Formul...
5,toro_rosso,Toro Rosso,Italian,http://en.wikipedia.org/wiki/Scuderia_Toro_Rosso


In [6]:
driver_standings_df = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/driver_standings.csv', index_col=0, na_values='\\N')
driver_standings_df.info()
driver_standings_df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 34863 entries, 1 to 73270
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   raceId        34863 non-null  int64  
 1   driverId      34863 non-null  int64  
 2   points        34863 non-null  float64
 3   position      34863 non-null  int64  
 4   positionText  34863 non-null  object 
 5   wins          34863 non-null  int64  
dtypes: float64(1), int64(4), object(1)
memory usage: 1.9+ MB


,raceId,driverId,points,position,positionText,wins
driverStandingsId,,,,,,
1,18,1,10.0,1,1,1
2,18,2,8.0,2,2,0
3,18,3,6.0,3,3,0
4,18,4,5.0,4,4,0
5,18,5,4.0,5,5,0


In [7]:
drivers_df = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/drivers.csv', index_col=0, na_values='\\N')
drivers_df.info()
drivers_df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 861 entries, 1 to 862
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   driverRef    861 non-null    object 
 1   number       59 non-null     float64
 2   code         104 non-null    object 
 3   forename     861 non-null    object 
 4   surname      861 non-null    object 
 5   dob          861 non-null    object 
 6   nationality  861 non-null    object 
 7   url          861 non-null    object 
dtypes: float64(1), object(7)
memory usage: 60.5+ KB


,driverRef,number,code,forename,surname,dob,nationality,url
driverId,,,,,,,,
1,hamilton,44.0,HAM,Lewis,Hamilton,1985-01-07,British,http://en.wikipedia.org/wiki/Lewis_Hamilton
2,heidfeld,NaN,HEI,Nick,Heidfeld,1977-05-10,German,http://en.wikipedia.org/wiki/Nick_Heidfeld
3,rosberg,6.0,ROS,Nico,Rosberg,1985-06-27,German,http://en.wikipedia.org/wiki/Nico_Rosberg
4,alonso,14.0,ALO,Fernando,Alonso,1981-07-29,Spanish,http://en.wikipedia.org/wiki/Fernando_Alonso
5,kovalainen,NaN,KOV,Heikki,Kovalainen,1981-10-19,Finnish,http://en.wikipedia.org/wiki/Heikki_Kovalainen


In [8]:
lap_times_df = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/lap_times.csv', index_col=0, na_values='\\N')
lap_times_df.info()
lap_times_df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 589081 entries, 841 to 1144
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   driverId      589081 non-null  int64 
 1   lap           589081 non-null  int64 
 2   position      589081 non-null  int64 
 3   time          589081 non-null  object
 4   milliseconds  589081 non-null  int64 
dtypes: int64(4), object(1)
memory usage: 27.0+ MB


,driverId,lap,position,time,milliseconds
raceId,,,,,
841,20,1,1,1:38.109,98109
841,20,2,1,1:33.006,93006
841,20,3,1,1:32.713,92713
841,20,4,1,1:32.803,92803
841,20,5,1,1:32.342,92342


In [9]:
pit_stops_df = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/pit_stops.csv', index_col=0, na_values='\\N')
pit_stops_df.info()
pit_stops_df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 11371 entries, 841 to 1144
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   driverId      11371 non-null  int64 
 1   stop          11371 non-null  int64 
 2   lap           11371 non-null  int64 
 3   time          11371 non-null  object
 4   duration      11371 non-null  object
 5   milliseconds  11371 non-null  int64 
dtypes: int64(4), object(2)
memory usage: 621.9+ KB


,driverId,stop,lap,time,duration,milliseconds
raceId,,,,,,
841,153,1,1,17:05:23,26.898,26898
841,30,1,1,17:05:52,25.021,25021
841,17,1,11,17:20:48,23.426,23426
841,4,1,12,17:22:34,23.251,23251
841,13,1,13,17:24:10,23.842,23842


In [10]:
qualifying_df = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/qualifying.csv', index_col=0, na_values='\\N')
qualifying_df.info()
qualifying_df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 10494 entries, 1 to 10551
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   raceId         10494 non-null  int64 
 1   driverId       10494 non-null  int64 
 2   constructorId  10494 non-null  int64 
 3   number         10494 non-null  int64 
 4   position       10494 non-null  int64 
 5   q1             10338 non-null  object
 6   q2             5847 non-null   object
 7   q3             3629 non-null   object
dtypes: int64(5), object(3)
memory usage: 737.9+ KB


,raceId,driverId,constructorId,number,position,q1,q2,q3
qualifyId,,,,,,,,
1,18,1,1,22,1,1:26.572,1:25.187,1:26.714
2,18,9,2,4,2,1:26.103,1:25.315,1:26.869
3,18,5,1,23,3,1:25.664,1:25.452,1:27.079
4,18,13,6,2,4,1:25.994,1:25.691,1:27.178
5,18,2,2,3,5,1:25.960,1:25.518,1:27.236


In [11]:
races_df = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/races.csv', index_col=0, na_values='\\N')
races_df.info()
races_df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 1125 entries, 1 to 1144
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   year         1125 non-null   int64 
 1   round        1125 non-null   int64 
 2   circuitId    1125 non-null   int64 
 3   name         1125 non-null   object
 4   date         1125 non-null   object
 5   time         394 non-null    object
 6   url          1125 non-null   object
 7   fp1_date     90 non-null     object
 8   fp1_time     68 non-null     object
 9   fp2_date     90 non-null     object
 10  fp2_time     68 non-null     object
 11  fp3_date     72 non-null     object
 12  fp3_time     53 non-null     object
 13  quali_date   90 non-null     object
 14  quali_time   68 non-null     object
 15  sprint_date  18 non-null     object
 16  sprint_time  15 non-null     object
dtypes: int64(3), object(14)
memory usage: 158.2+ KB


,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
raceId,,,,,,,,,,,,,,,,,
1,2009,1,1,Australian Grand Prix,2009-03-29,06:00:00,http://en.wikipedia.org/wiki/2009_Australian_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009,2,2,Malaysian Grand Prix,2009-04-05,09:00:00,http://en.wikipedia.org/wiki/2009_Malaysian_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2009,3,17,Chinese Grand Prix,2009-04-19,07:00:00,http://en.wikipedia.org/wiki/2009_Chinese_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2009,4,3,Bahrain Grand Prix,2009-04-26,12:00:00,http://en.wikipedia.org/wiki/2009_Bahrain_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2009,5,4,Spanish Grand Prix,2009-05-10,12:00:00,http://en.wikipedia.org/wiki/2009_Spanish_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
results_df = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/results.csv', index_col=0, na_values='\\N')
results_df.info()
results_df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 26759 entries, 1 to 26764
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   raceId           26759 non-null  int64  
 1   driverId         26759 non-null  int64  
 2   constructorId    26759 non-null  int64  
 3   number           26753 non-null  float64
 4   grid             26759 non-null  int64  
 5   position         15806 non-null  float64
 6   positionText     26759 non-null  object 
 7   positionOrder    26759 non-null  int64  
 8   points           26759 non-null  float64
 9   laps             26759 non-null  int64  
 10  time             7680 non-null   object 
 11  milliseconds     7680 non-null   float64
 12  fastestLap       8252 non-null   float64
 13  rank             8510 non-null   float64
 14  fastestLapTime   8252 non-null   object 
 15  fastestLapSpeed  8252 non-null   float64
 16  statusId         26759 non-null  int64  
dtypes: float64(7), in

,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
resultId,,,,,,,,,,,,,,,,,
1,18,1,1,22.0,1,1.0,1,1,10.0,58,1:34:50.616,5690616.0,39.0,2.0,1:27.452,218.300,1
2,18,2,2,3.0,5,2.0,2,2,8.0,58,+5.478,5696094.0,41.0,3.0,1:27.739,217.586,1
3,18,3,3,7.0,7,3.0,3,3,6.0,58,+8.163,5698779.0,41.0,5.0,1:28.090,216.719,1
4,18,4,4,5.0,11,4.0,4,4,5.0,58,+17.181,5707797.0,58.0,7.0,1:28.603,215.464,1
5,18,5,1,23.0,3,5.0,5,5,4.0,58,+18.014,5708630.0,43.0,1.0,1:27.418,218.385,1


In [13]:
seasons_df = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/seasons.csv', index_col=0, na_values='\\N')
seasons_df.info()
seasons_df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 75 entries, 2009 to 2024
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   url     75 non-null     object
dtypes: object(1)
memory usage: 1.2+ KB


,url
year,
2009,http://en.wikipedia.org/wiki/2009_Formula_One_...
2008,http://en.wikipedia.org/wiki/2008_Formula_One_...
2007,http://en.wikipedia.org/wiki/2007_Formula_One_...
2006,http://en.wikipedia.org/wiki/2006_Formula_One_...
2005,http://en.wikipedia.org/wiki/2005_Formula_One_...


In [14]:
sprint_results_df = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/sprint_results.csv', index_col=0, na_values='\\N')
sprint_results_df.info()
sprint_results_df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 360 entries, 1 to 360
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   raceId          360 non-null    int64  
 1   driverId        360 non-null    int64  
 2   constructorId   360 non-null    int64  
 3   number          360 non-null    int64  
 4   grid            360 non-null    int64  
 5   position        345 non-null    float64
 6   positionText    360 non-null    object 
 7   positionOrder   360 non-null    int64  
 8   points          360 non-null    int64  
 9   laps            360 non-null    int64  
 10  time            340 non-null    object 
 11  milliseconds    340 non-null    float64
 12  fastestLap      351 non-null    float64
 13  fastestLapTime  351 non-null    object 
 14  statusId        360 non-null    int64  
dtypes: float64(3), int64(9), object(3)
memory usage: 45.0+ KB


,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,fastestLapTime,statusId
resultId,,,,,,,,,,,,,,,
1,1061,830,9,33,2,1.0,1,1,3,17,25:38.426,1538426.0,14.0,1:30.013,1
2,1061,1,131,44,1,2.0,2,2,2,17,+1.430,1539856.0,17.0,1:29.937,1
3,1061,822,131,77,3,3.0,3,3,1,17,+7.502,1545928.0,17.0,1:29.958,1
4,1061,844,6,16,4,4.0,4,4,0,17,+11.278,1549704.0,16.0,1:30.163,1
5,1061,846,1,4,6,5.0,5,5,0,17,+24.111,1562537.0,16.0,1:30.566,1


In [15]:
status_df = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/status.csv', index_col=0, na_values='\\N')
status_df.info()
status_df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 139 entries, 1 to 141
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   status  139 non-null    object
dtypes: object(1)
memory usage: 2.2+ KB


,status
statusId,
1,Finished
2,Disqualified
3,Accident
4,Collision
5,Engine
